# mars_walk_forward_validation_tuning

MARS-style classifier for the same rich-price three-class experiment setup used by the other model notebooks.

The notebook avoids an extra `sklearn-contrib-py-earth` dependency and instead uses a deterministic MARS-style hinge basis expansion followed by regularized multinomial logistic regression. The validation and final-test protocol is the same as in the other classical model notebooks.

In [1]:
from __future__ import annotations

from itertools import combinations
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 260)

warnings.filterwarnings(
    'ignore',
    message=r"'penalty' was deprecated.*",
    category=FutureWarning,
    module=r'sklearn\.linear_model\._logistic',
)


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'datasets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.experiment_config import build_default_config
from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.model_report_builder import ModelReportBuilder
from notebook_utils.split_utils import make_split_dates, make_walk_forward_fold_specs, subset_by_dates

CONFIG = build_default_config(PROJECT_ROOT)


def format_grid_value(value: object) -> str:
    if value is None:
        return 'none'
    if isinstance(value, float):
        return f'{value:g}'.replace('.', 'p')
    return str(value).replace('.', 'p')


MARS_PARAM_GRID = [
    {
        'param_set': 'mars_q2_d1_i0_l1_C0p1_balanced',
        'n_knots': 2,
        'interaction_degree': 1,
        'max_interaction_terms': 0,
        'penalty': 'l1',
        'C': 0.1,
        'solver': 'saga',
        'class_weight': 'balanced',
        'max_iter': 5000,
    },
    {
        'param_set': 'mars_q3_d1_i0_l1_C0p1_balanced',
        'n_knots': 3,
        'interaction_degree': 1,
        'max_interaction_terms': 0,
        'penalty': 'l1',
        'C': 0.1,
        'solver': 'saga',
        'class_weight': 'balanced',
        'max_iter': 5000,
    },
    {
        'param_set': 'mars_q3_d1_i0_l1_C0p5_balanced',
        'n_knots': 3,
        'interaction_degree': 1,
        'max_interaction_terms': 0,
        'penalty': 'l1',
        'C': 0.5,
        'solver': 'saga',
        'class_weight': 'balanced',
        'max_iter': 5000,
    },
    {
        'param_set': 'mars_q5_d1_i0_l1_C0p1_balanced',
        'n_knots': 5,
        'interaction_degree': 1,
        'max_interaction_terms': 0,
        'penalty': 'l1',
        'C': 0.1,
        'solver': 'saga',
        'class_weight': 'balanced',
        'max_iter': 5000,
    },
    {
        'param_set': 'mars_q2_d2_i32_l1_C0p1_balanced',
        'n_knots': 2,
        'interaction_degree': 2,
        'max_interaction_terms': 32,
        'penalty': 'l1',
        'C': 0.1,
        'solver': 'saga',
        'class_weight': 'balanced',
        'max_iter': 5000,
    },
    {
        'param_set': 'mars_q3_d2_i48_l1_C0p1_balanced',
        'n_knots': 3,
        'interaction_degree': 2,
        'max_interaction_terms': 48,
        'penalty': 'l1',
        'C': 0.1,
        'solver': 'saga',
        'class_weight': 'balanced',
        'max_iter': 5000,
    },
    {
        'param_set': 'mars_q3_d1_i0_l2_C0p3_balanced',
        'n_knots': 3,
        'interaction_degree': 1,
        'max_interaction_terms': 0,
        'penalty': 'l2',
        'C': 0.3,
        'solver': 'lbfgs',
        'class_weight': 'balanced',
        'max_iter': 3000,
    },
]

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG['max_features_per_model'],
)

PRICE_FEATURES = feature_grid.price_features
VOLUME_FEATURE_OPTIONS = feature_grid.volume_feature_options
GDELT_FEATURE_OPTIONS = feature_grid.gdelt_feature_options
GDELT_SENTIMENT_FEATURE_OPTIONS = feature_grid.gdelt_sentiment_feature_options
GDELT_ATTENTION_FEATURE_OPTIONS = feature_grid.gdelt_attention_feature_options
REDDIT_FEATURE_OPTIONS = feature_grid.reddit_feature_options
REDDIT_ATTENTION_FEATURE_OPTIONS = feature_grid.reddit_attention_feature_options
GOOGLE_TRENDS_FEATURE_OPTIONS = feature_grid.google_trends_feature_options
GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = feature_grid.google_score_attention_feature_options
DERIVED_FEATURE_COLUMNS = feature_grid.derived_feature_columns
BASE_VOLUME_OPTION = feature_grid.base_volume_option
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SET_SPECS = feature_grid.feature_set_specs
FEATURE_SETS = feature_grid.feature_sets
FEATURE_SET_METADATA = feature_grid.feature_set_metadata
SKIPPED_FEATURE_SETS = feature_grid.skipped_feature_sets
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test

pd.DataFrame(MARS_PARAM_GRID)


,param_set,n_knots,interaction_degree,max_interaction_terms,penalty,C,solver,class_weight,max_iter
0,mars_q2_d1_i0_l1_C0p1_balanced,2,1,0,l1,0.1,saga,balanced,5000
1,mars_q3_d1_i0_l1_C0p1_balanced,3,1,0,l1,0.1,saga,balanced,5000
2,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000
3,mars_q5_d1_i0_l1_C0p1_balanced,5,1,0,l1,0.1,saga,balanced,5000
4,mars_q2_d2_i32_l1_C0p1_balanced,2,2,32,l1,0.1,saga,balanced,5000
5,mars_q3_d2_i48_l1_C0p1_balanced,3,2,48,l1,0.1,saga,balanced,5000
6,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000


In [3]:
class MarsHingeBasisTransformer(BaseEstimator, TransformerMixin):
    """Build a compact MARS-style hinge basis from numeric tabular features."""

    def __init__(
        self,
        n_knots: int = 3,
        knot_quantile_low: float = 0.1,
        knot_quantile_high: float = 0.9,
        include_linear_terms: bool = True,
        interaction_degree: int = 1,
        max_interaction_terms: int = 0,
    ):
        self.n_knots = n_knots
        self.knot_quantile_low = knot_quantile_low
        self.knot_quantile_high = knot_quantile_high
        self.include_linear_terms = include_linear_terms
        self.interaction_degree = interaction_degree
        self.max_interaction_terms = max_interaction_terms

    @staticmethod
    def _as_array(X) -> np.ndarray:
        X_array = np.asarray(X, dtype=float)
        if X_array.ndim != 2:
            raise ValueError(f'Expected a 2D feature matrix, got shape={X_array.shape}')
        return X_array

    def fit(self, X, y=None):
        X_array = self._as_array(X)
        self.n_features_in_ = X_array.shape[1]
        quantiles = np.linspace(float(self.knot_quantile_low), float(self.knot_quantile_high), int(self.n_knots))

        self.univariate_basis_specs_ = []
        for feature_idx in range(self.n_features_in_):
            values = X_array[:, feature_idx]
            values = values[np.isfinite(values)]
            if values.size == 0:
                continue
            knots = np.unique(np.quantile(values, quantiles))
            for knot in knots:
                self.univariate_basis_specs_.append((feature_idx, float(knot), 'right'))
                self.univariate_basis_specs_.append((feature_idx, float(knot), 'left'))

        univariate_basis = self._transform_univariate(X_array)
        self.interaction_specs_ = []
        if int(self.interaction_degree) >= 2 and int(self.max_interaction_terms) > 0:
            self.interaction_specs_ = self._select_interactions(univariate_basis, y)

        linear_terms = self.n_features_in_ if self.include_linear_terms else 0
        self.n_output_features_ = linear_terms + len(self.univariate_basis_specs_) + len(self.interaction_specs_)
        return self

    def transform(self, X):
        X_array = self._as_array(X)
        if X_array.shape[1] != self.n_features_in_:
            raise ValueError(f'Expected {self.n_features_in_} features, got {X_array.shape[1]}')

        parts = []
        if self.include_linear_terms:
            parts.append(X_array)

        univariate_basis = self._transform_univariate(X_array)
        if univariate_basis.shape[1] > 0:
            parts.append(univariate_basis)

        if self.interaction_specs_:
            interaction_basis = np.column_stack(
                [univariate_basis[:, left_idx] * univariate_basis[:, right_idx] for left_idx, right_idx in self.interaction_specs_]
            )
            parts.append(interaction_basis)

        if not parts:
            return np.empty((X_array.shape[0], 0))
        return np.hstack(parts)

    def _transform_univariate(self, X_array: np.ndarray) -> np.ndarray:
        if not self.univariate_basis_specs_:
            return np.empty((X_array.shape[0], 0))

        basis = np.empty((X_array.shape[0], len(self.univariate_basis_specs_)), dtype=float)
        for basis_idx, (feature_idx, knot, direction) in enumerate(self.univariate_basis_specs_):
            feature_values = X_array[:, feature_idx]
            if direction == 'right':
                basis[:, basis_idx] = np.maximum(0.0, feature_values - knot)
            else:
                basis[:, basis_idx] = np.maximum(0.0, knot - feature_values)
        return basis

    def _select_interactions(self, univariate_basis: np.ndarray, y) -> list[tuple[int, int]]:
        max_terms = int(self.max_interaction_terms)
        if max_terms <= 0 or univariate_basis.shape[1] < 2:
            return []

        feature_by_basis = [spec[0] for spec in self.univariate_basis_specs_]
        scored_pairs = []
        for left_idx, right_idx in combinations(range(univariate_basis.shape[1]), 2):
            if feature_by_basis[left_idx] == feature_by_basis[right_idx]:
                continue
            values = univariate_basis[:, left_idx] * univariate_basis[:, right_idx]
            score = self._score_candidate(values, y)
            if score > 0.0:
                scored_pairs.append((score, left_idx, right_idx))

        scored_pairs.sort(key=lambda item: (-item[0], item[1], item[2]))
        return [(left_idx, right_idx) for _, left_idx, right_idx in scored_pairs[:max_terms]]

    @staticmethod
    def _score_candidate(values: np.ndarray, y) -> float:
        centered_values = values - values.mean()
        values_norm = np.linalg.norm(centered_values)
        if values_norm <= 0.0:
            return 0.0
        if y is None:
            return float(values_norm)

        y_array = np.asarray(y)
        best_score = 0.0
        for class_value in np.unique(y_array):
            target = (y_array == class_value).astype(float)
            centered_target = target - target.mean()
            target_norm = np.linalg.norm(centered_target)
            if target_norm <= 0.0:
                continue
            score = abs(float(centered_values @ centered_target) / float(values_norm * target_norm))
            best_score = max(best_score, score)
        return best_score


def build_mars_pipeline_from_params(params: dict) -> Pipeline:
    mars_params = {
        'n_knots': int(params['n_knots']),
        'interaction_degree': int(params['interaction_degree']),
        'max_interaction_terms': int(params['max_interaction_terms']),
        'include_linear_terms': True,
    }
    logreg_params = {
        'penalty': params['penalty'],
        'C': float(params['C']),
        'solver': params['solver'],
        'class_weight': params.get('class_weight'),
        'max_iter': int(params['max_iter']),
        'random_state': CONFIG['random_state'],
    }
    return Pipeline(
        [
            ('imputer', SimpleImputer(strategy='median')),
            ('input_scaler', StandardScaler()),
            ('mars_basis', MarsHingeBasisTransformer(**mars_params)),
            ('basis_scaler', StandardScaler()),
            ('model', LogisticRegression(**logreg_params)),
        ]
    )


In [4]:
raw_df = pd.read_csv(CONFIG['dataset_path'], parse_dates=['date'])
raw_df = raw_df[~raw_df['ticker'].isin(CONFIG['excluded_tickers'])].copy()
raw_df = raw_df.sort_values(['ticker', 'date']).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG['neutral_band'])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG['test_size'],
    validation_fraction_within_pretest=CONFIG['validation_fraction_within_pretest'],
    min_validation_dates=CONFIG['min_validation_dates'],
    gap_days=CONFIG['gap_days'],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG['walk_forward_folds'],
    validation_size=CONFIG['walk_forward_validation_dates'],
    min_train_dates=CONFIG['walk_forward_min_train_dates'],
    gap_days=CONFIG['gap_days'],
)

split_date_map = {
    'train': train_dates,
    'validation': validation_dates,
    'test': test_dates,
}
feature_df['split'] = 'gap'
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df['date'].isin(split_dates), 'split'] = split_name

modeled_df = feature_df[feature_df['target'].isin(ClassificationMetrics.CLASS_VALUES)].copy()
modeled_df['target'] = modeled_df['target'].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)
train_validation_df = subset_by_dates(modeled_df, list(train_dates) + list(validation_dates))

split_summary_rows = []
for split_name in ['train', 'validation', 'test']:
    all_split_df = feature_df[feature_df['split'].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df['split'].eq(split_name)]
    class_rates = modeled_split_df['target'].value_counts(normalize=True)
    split_summary_rows.append(
        {
            'split': split_name,
            'session_rows': len(all_split_df),
            'modeled_rows': len(modeled_split_df),
            'session_dates': all_split_df['date'].nunique(),
            'modeled_dates': modeled_split_df['date'].nunique(),
            'date_min': all_split_df['date'].min(),
            'date_max': all_split_df['date'].max(),
            'target_down_rate': float(class_rates.get(0, 0.0)),
            'target_neutral_rate': float(class_rates.get(1, 0.0)),
            'target_up_rate': float(class_rates.get(2, 0.0)),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            'fold': spec['fold'],
            'train_n_dates': spec['train_n_dates'],
            'train_date_min': spec['train_date_min'],
            'train_date_max': spec['train_date_max'],
            'validation_n_dates': spec['validation_n_dates'],
            'validation_date_min': spec['validation_date_min'],
            'validation_date_max': spec['validation_date_max'],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df


,split,session_rows,modeled_rows,session_dates,modeled_dates,date_min,date_max,target_down_rate,target_neutral_rate,target_up_rate
0,train,5632,5632,704,704,2021-01-04,2023-10-19,0.386541,0.208629,0.404830
1,validation,1880,1880,235,235,2023-10-23,2024-09-27,0.327128,0.242553,0.430319
2,test,2512,2504,314,313,2024-10-01,2025-12-31,0.356629,0.246805,0.396565


In [5]:
walk_forward_fold_summary_df


,fold,train_n_dates,train_date_min,train_date_max,validation_n_dates,validation_date_min,validation_date_max
0,1,383,2021-01-04,2022-07-12,80,2022-07-14,2022-11-03
1,2,463,2021-01-04,2022-11-02,80,2022-11-04,2023-03-02
2,3,543,2021-01-04,2023-03-01,80,2023-03-03,2023-06-27
3,4,623,2021-01-04,2023-06-26,80,2023-06-28,2023-10-19


In [6]:
neutral_summary_by_ticker_df = (
    feature_df.groupby('ticker')
    .agg(
        rows=('target_available', 'size'),
        target_available=('target_available', 'sum'),
        neutral=('is_neutral', 'sum'),
    )
    .reset_index()
)
neutral_summary_by_ticker_df['neutral_rate_among_available'] = (
    neutral_summary_by_ticker_df['neutral'] / neutral_summary_by_ticker_df['target_available']
)
neutral_summary_by_ticker_df['modeled_rate_among_available'] = 1.0 - neutral_summary_by_ticker_df['neutral_rate_among_available']

neutral_summary_by_split_df = (
    feature_df[feature_df['split'].isin(['train', 'validation', 'test'])]
    .groupby('split')
    .agg(
        rows=('target_available', 'size'),
        target_available=('target_available', 'sum'),
        neutral=('is_neutral', 'sum'),
    )
    .reindex(['train', 'validation', 'test'])
    .reset_index()
)
neutral_summary_by_split_df['neutral_rate_among_available'] = (
    neutral_summary_by_split_df['neutral'] / neutral_summary_by_split_df['target_available']
)
neutral_summary_by_split_df['modeled_rate_among_available'] = 1.0 - neutral_summary_by_split_df['neutral_rate_among_available']

print('Neutral coverage by split')
print(neutral_summary_by_split_df.to_string(index=False))
print('\nNeutral coverage by ticker')
neutral_summary_by_ticker_df


Neutral coverage by split
     split  rows  target_available  neutral  neutral_rate_among_available  modeled_rate_among_available
     train  5632              5632     1175                      0.208629                      0.791371
validation  1880              1880      456                      0.242553                      0.757447
      test  2512              2504      618                      0.246805                      0.753195

Neutral coverage by ticker


,ticker,rows,target_available,neutral,neutral_rate_among_available,modeled_rate_among_available
0,AAPL,1255,1254,378,0.301435,0.698565
1,AMD,1255,1254,220,0.175439,0.824561
2,AMZN,1255,1254,300,0.239234,0.760766
3,GOOGL,1255,1254,319,0.254386,0.745614
4,META,1255,1254,277,0.220893,0.779107
5,MSFT,1255,1254,400,0.318979,0.681021
6,NVDA,1255,1254,173,0.137959,0.862041
7,TSLA,1255,1254,184,0.146730,0.853270


In [7]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f'Unknown feature sets: {missing_requested_feature_sets}')

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f'Missing feature columns: {missing_feature_columns}')

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            'feature_set': feature_set_name,
            'feature_family': FEATURE_SET_METADATA[feature_set_name]['feature_family'],
            'n_features': len(features),
            'features': features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(['feature_family', 'n_features', 'feature_set']).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print('Selection metric:', CONFIG['selection_metric'])
print('Primary validation metric:', CONFIG['primary_validation_metric'])
print('Walk-forward folds:', len(walk_forward_fold_specs))
print('Target classes:', ClassificationMetrics.CLASS_LABELS)
print('Prediction rule: multiclass probability argmax')
print('Max features per model:', CONFIG['max_features_per_model'])
print('Feature sets to test:', len(FEATURE_SETS_TO_TEST))
print('Attention feature sets:', sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST))
print('Skipped feature sets above max feature limit:', len(SKIPPED_FEATURE_SETS))
print('MARS-style parameter sets:', len(MARS_PARAM_GRID))
print('Walk-forward validation fits:', len(FEATURE_SETS_TO_TEST) * len(MARS_PARAM_GRID) * len(walk_forward_fold_specs))

candidate_feature_sets_df


Selection metric: balanced_accuracy
Primary validation metric: balanced_accuracy
Walk-forward folds: 4
Target classes: {0: 'down', 1: 'neutral', 2: 'up'}
Prediction rule: multiclass probability argmax
Max features per model: 14
Feature sets to test: 80
Attention feature sets: 38
Skipped feature sets above max feature limit: 0
MARS-style parameter sets: 7
Walk-forward validation fits: 2240


,feature_set,feature_family,n_features,features
0,Model B - price + volume | volume log1p zscore...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
1,Model B - price + volume | volume percentile r...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
2,Model B - price + volume | volume zscore 10d,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
3,Model B - price + volume | volume zscore 20d,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
4,Model B - price + volume | volume zscore 20d c...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
...,...,...,...,...
75,Model F - price + volume + all alternative dat...,price + volume + all alternative data,14,"[return_1d, return_5d, return_20d, rolling_vol..."
76,Model N - price + volume + all attention | per...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."
77,Model N - price + volume + all attention | zsc...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."
78,Model N - price + volume + all attention | zsc...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."


In [8]:
MARS_PARAM_COLUMNS = [
    'n_knots',
    'interaction_degree',
    'max_interaction_terms',
    'penalty',
    'C',
    'solver',
    'class_weight',
    'max_iter',
    'basis_terms',
    'selected_interactions',
]
MARS_PARAM_COLUMNS_FOR_DISPLAY = MARS_PARAM_COLUMNS


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if 'google_trends_above_ticker_train_median' in features:
        return FeatureFrameBuilder.add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def params_from_result_row(row: dict | pd.Series) -> dict:
    class_weight = row.get('class_weight', None)
    if pd.isna(class_weight):
        class_weight = None
    return {
        'param_set': row['param_set'],
        'n_knots': int(row['n_knots']),
        'interaction_degree': int(row['interaction_degree']),
        'max_interaction_terms': int(row['max_interaction_terms']),
        'penalty': row['penalty'],
        'C': float(row['C']),
        'solver': row['solver'],
        'class_weight': class_weight,
        'max_iter': int(row['max_iter']),
    }


def add_param_columns(row: dict, params: dict, param_columns: list[str]) -> None:
    for column in param_columns:
        row[column] = params.get(column)


def evaluate_mars_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )

    pipeline = build_mars_pipeline_from_params(params)
    pipeline.fit(train_features_df[features], train_features_df['target'])
    probabilities = pipeline.predict_proba(eval_features_df[features])
    classes = pipeline.named_steps['model'].classes_
    preds = ClassificationMetrics.predictions_from_probabilities(probabilities, classes)

    metric_result = ClassificationMetrics.metrics_from_predictions(eval_features_df['target'], preds)
    preds = metric_result.pop('preds')
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    mars_basis = pipeline.named_steps['mars_basis']
    row = {
        'split': split_name,
        'feature_set': feature_set_name,
        'feature_family': metadata.get('feature_family'),
        'param_set': params['param_set'],
        'n_features': len(features),
        **metric_result,
    }
    add_param_columns(row, params, MARS_PARAM_COLUMNS_FOR_DISPLAY)
    row['basis_terms'] = int(mars_basis.n_output_features_)
    row['selected_interactions'] = int(len(mars_basis.interaction_specs_))

    if not return_predictions:
        return row

    predictions_df = eval_features_df[['date', 'ticker', 'target']].copy()
    for column, values in ClassificationMetrics.probability_column_dict(probabilities, classes).items():
        predictions_df[column] = values
    predictions_df['prediction'] = preds
    return row, predictions_df


def evaluate_mars_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_mars_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec['train_dates']),
            eval_input_df=subset_by_dates(modeled_input_df, spec['validation_dates']),
            split_name='walk_forward_validation',
        )
        fold_rows.append(
            {
                **fold_row,
                'fold': spec['fold'],
                'fold_train_n_dates': spec['train_n_dates'],
                'fold_validation_n_dates': spec['validation_n_dates'],
                'fold_train_date_min': spec['train_date_min'],
                'fold_train_date_max': spec['train_date_max'],
                'fold_validation_date_min': spec['validation_date_min'],
                'fold_validation_date_max': spec['validation_date_max'],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    metric_means = {
        metric: float(fold_results_df[metric].mean())
        for metric in ['accuracy', 'balanced_accuracy', 'f1_score', 'f1_weighted']
    }
    summary_row = {
        'split': 'walk_forward_validation',
        'feature_set': feature_set_name,
        'feature_family': metadata.get('feature_family'),
        'param_set': params['param_set'],
        'n_features': len(features),
        **metric_means,
    }
    add_param_columns(summary_row, params, MARS_PARAM_COLUMNS_FOR_DISPLAY)
    summary_row['basis_terms'] = float(fold_results_df['basis_terms'].mean())
    summary_row['selected_interactions'] = float(fold_results_df['selected_interactions'].mean())
    return summary_row, fold_rows


In [9]:
selection_metric = CONFIG['selection_metric']
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in MARS_PARAM_GRID:
        summary_row, fold_rows = evaluate_mars_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f'Selection metric is not available: {selection_metric}')

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, 'balanced_accuracy', 'f1_score', 'accuracy', 'feature_set', 'param_set'],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)


C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [10]:
validation_best_by_feature_set_df = ModelReportBuilder.select_best_validation_by_feature_set(
    validation_grid_results_df,
    selection_metric=CONFIG['selection_metric'],
)

validation_best_by_feature_set_report_df = ModelReportBuilder.build_validation_best_by_feature_set_report(
    validation_best_by_feature_set_df,
    param_columns=MARS_PARAM_COLUMNS_FOR_DISPLAY,
)

validation_best_by_feature_set_report_df


,feature_family,feature_set,n_features,param_set,n_knots,interaction_degree,max_interaction_terms,penalty,C,solver,class_weight,max_iter,basis_terms,selected_interactions,validation_accuracy,validation_balanced_accuracy,validation_f1_score,validation_f1_weighted
0,price + volume + GDELT,Model C - price + volume + GDELT | GDELT zscor...,11,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,77.0,0.0,0.360938,0.375180,0.345516,0.351701
1,price + volume + GDELT,Model C - price + volume + GDELT | GDELT zscor...,12,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,80.0,0.0,0.361328,0.374552,0.346302,0.353478
2,price + volume + GDELT,Model C - price + volume + GDELT | GDELT zscor...,11,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,77.0,0.0,0.361328,0.374077,0.345762,0.353261
3,price + volume + GDELT + Google,Model H - price + volume + GDELT + Google | zs...,12,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,84.0,0.0,0.360547,0.373965,0.345701,0.352722
4,price + volume + GDELT + Reddit attention,Model M - price + volume + GDELT + Reddit atte...,11,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,77.0,0.0,0.359375,0.373406,0.344513,0.350836
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,price + volume + GDELT attention lags,Model S - price + volume + GDELT attention lag...,12,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,84.0,0.0,0.342969,0.356286,0.328179,0.333192
76,price + volume + Google score attention,Model L - price + volume + Google score attent...,12,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,84.0,0.0,0.342578,0.354920,0.328912,0.333578
77,price + volume + Google attention lags,Model U - price + volume + Google attention la...,12,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,84.0,0.0,0.342578,0.354920,0.328912,0.333578
78,price + volume + GDELT sentiment lags + Google...,Model X - price + volume + GDELT sentiment lag...,14,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,98.0,0.0,0.343359,0.354585,0.329537,0.335827


In [11]:
best_validation_params_df = validation_best_by_feature_set_df.copy()


In [12]:
validation_refit_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient='records'):
    params = params_from_result_row(row)
    feature_set_name = row['feature_set']
    validation_refit_row = evaluate_mars_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name='validation_refit_train',
    )
    validation_refit_rows.append(validation_refit_row)
    test_rows.append(
        evaluate_mars_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_validation_df,
            eval_input_df=test_df,
            split_name='test_refit_train_validation',
        )
    )

validation_refit_results_df = pd.DataFrame(validation_refit_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ['balanced_accuracy', 'f1_score', 'accuracy', 'feature_set'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

(
    simple_hyperparameter_summary_df,
    baseline_walk_forward_row,
    baseline_validation_refit_row,
    baseline_test_row,
) = ModelReportBuilder.build_simple_hyperparameter_summary(
    best_validation_params_df=best_validation_params_df,
    validation_refit_results_df=validation_refit_results_df,
    test_best_validation_params_df=test_best_validation_params_df,
    baseline_feature_set=BASELINE_FEATURE_SET,
)


C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [13]:
validation_selected_family_test_report_df = ModelReportBuilder.build_validation_selected_family_test_report(
    simple_hyperparameter_summary_df,
    param_columns=MARS_PARAM_COLUMNS_FOR_DISPLAY,
)

final_test_verification_path = ModelReportBuilder.save_final_test_verification(
    validation_selected_family_test_report_df,
    model_name='mars',
    output_dir=PROJECT_ROOT / 'notebooks' / 'outputs-three-classes-rich-price',
)
print(f'Saved final test verification to: {final_test_verification_path}')

validation_selected_family_test_report_df


Saved final test verification to: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks\outputs-three-classes-rich-price\mars_final_test_verification.csv


,feature_family,feature_set,n_features,param_set,n_knots,interaction_degree,max_interaction_terms,penalty,C,solver,class_weight,max_iter,basis_terms,selected_interactions,validation_accuracy,validation_balanced_accuracy,validation_f1_score,validation_f1_weighted,test_accuracy,test_balanced_accuracy,test_f1_score,test_f1_weighted,price_volume_baseline_feature_set,price_volume_baseline_test_balanced_accuracy,test_balanced_accuracy_change_vs_price_volume
0,price + volume + Reddit attention,Model K - price + volume + Reddit attention | ...,10,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,70.0,0.0,0.354688,0.368917,0.340310,0.345751,0.379792,0.411719,0.376585,0.369512,Model B - price + volume | volume zscore 20d,0.407926,0.003793
1,price + volume + Reddit attention lags,Model T - price + volume + Reddit attention la...,12,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,84.0,0.0,0.347266,0.360064,0.330411,0.336888,0.376597,0.409479,0.372946,0.365051,Model B - price + volume | volume zscore 20d,0.407926,0.001553
2,price + volume + Reddit + Google,Model I - price + volume + Reddit + Google | l...,12,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,84.0,0.0,0.350391,0.364660,0.336688,0.341948,0.378195,0.408656,0.375593,0.368270,Model B - price + volume | volume zscore 20d,0.407926,0.000730
3,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,12,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,84.0,0.0,0.355078,0.369292,0.340468,0.346413,0.376597,0.407940,0.373709,0.365799,Model B - price + volume | volume zscore 20d,0.407926,0.000014
4,price + volume,Model B - price + volume | volume zscore 20d,9,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,63.0,0.0,0.351953,0.366821,0.336970,0.342101,0.375000,0.407926,0.371251,0.363359,Model B - price + volume | volume zscore 20d,0.407926,0.000000
5,price only,Model A - price only,8,mars_q3_d1_i0_l2_C0p3_balanced,3,1,0,l2,0.3,lbfgs,balanced,3000,56.0,0.0,0.350000,0.365715,0.336476,0.341059,0.373003,0.406859,0.368804,0.360302,Model B - price + volume | volume zscore 20d,0.407926,-0.001067
6,price + volume + GDELT sentiment + Reddit atte...,Model O - price + volume + GDELT sentiment + R...,11,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,77.0,0.0,0.356641,0.370049,0.341261,0.347593,0.373802,0.405365,0.370677,0.363047,Model B - price + volume | volume zscore 20d,0.407926,-0.002561
7,price + volume + GDELT + Reddit attention,Model M - price + volume + GDELT + Reddit atte...,11,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,77.0,0.0,0.359375,0.373406,0.344513,0.350836,0.373003,0.404499,0.369492,0.362407,Model B - price + volume | volume zscore 20d,0.407926,-0.003427
8,price + volume + GDELT sentiment lags + Reddit...,Model V - price + volume + GDELT sentiment lag...,14,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,98.0,0.0,0.352344,0.366340,0.337404,0.342816,0.371805,0.403639,0.368433,0.361141,Model B - price + volume | volume zscore 20d,0.407926,-0.004287
9,price + volume + all alternative data,Model F - price + volume + all alternative dat...,14,mars_q3_d1_i0_l1_C0p5_balanced,3,1,0,l1,0.5,saga,balanced,5000,98.0,0.0,0.356250,0.368464,0.342442,0.349839,0.373802,0.403365,0.371475,0.364371,Model B - price + volume | volume zscore 20d,0.407926,-0.004561
